# 03 — Induction Heads

The "hello world" circuit of mech interp. An induction head implements:
*if token `A` was followed by `B` earlier in the context, and I now see `A`
again, boost the prediction of `B`.* This is a genuinely learned algorithm,
not a hard-coded rule, and it's present in essentially every transformer
language model.

It's actually a **two-head circuit**:
1. A **previous-token head** in an early layer writes "what token came right
   before me" into each position's residual stream.
2. An **induction head** in a later layer attends to the position *after*
   the last occurrence of the current token, using the previous-token head's
   output to find it, and copies that token forward.


## Shorthand in this notebook

- **`seq_len`** — sequence length; here specifically the length of the
  random block that gets repeated (not the full prompt length, which is
  `2 * seq_len + 1` once you count the one-token prefix).
- **Layer.Head notation (`5.5`, `L5H5`)** — you'll see mech interp papers
  refer to "head 5.5" or "L5H5" — both mean "layer 5, head 5." When the top
  result below prints `layer 5 head 5`, that's the head the field commonly
  calls **5.5**, one of the two canonical induction heads originally found
  in GPT-2 small.
- **`diagonal(offset=...)`** — not mech-interp-specific, just a PyTorch/NumPy
  method: `tensor.diagonal(offset=k)` reads the diagonal that's `k` steps
  above (positive) or below (negative) the main diagonal. Used here as a
  fast way to grab "attention paid from position `p` to position `p -
  seq_len`" for every `p` at once, instead of looping.


In [1]:
import torch
from transformer_lens import HookedTransformer, utils
import circuitsvis as cv
import plotly.express as px

torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2", device=device)


/home/keqingsimp/.conda/envs/mech-interp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/tmp/ipykernel_68136/1105619070.py:2: DeprecationWarning: The 'utils' module has been deprecated. Please use 'transformer_lens.utilities' instead. Importing from utils.py will be removed in TransformerLens 4.0.
  from transformer_lens import HookedTransformer, utils


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 17132.37it/s]

Loaded pretrained model gpt2 into HookedTransformer


## Build a synthetic test: repeated random tokens

If we feed the model a random sequence and then *repeat it*, a perfect
induction head should nail the prediction on the repeat (since it saw the
exact continuation once already).


In [2]:
def induction_batch(model, seq_len=25, batch=4, seed=0):
    torch.manual_seed(seed)
    prefix = torch.randint(0, model.cfg.d_vocab, (batch, 1))
    half = torch.randint(0, model.cfg.d_vocab, (batch, seq_len))
    tokens = torch.cat([prefix, half, half], dim=1).to(device)
    return tokens

tokens = induction_batch(model)
logits, cache = model.run_with_cache(tokens)
print(tokens.shape)


torch.Size([4, 51])


## Induction score per head

For each head, measure: at each position in the *second* half, how much
attention probability goes to the position exactly `seq_len` steps back
(i.e. the token that followed this same token last time)? Average this over
all positions and batch elements to get one score per head.


In [3]:
def induction_scores(cache, seq_len, n_layers, n_heads):
    scores = torch.zeros(n_layers, n_heads)
    for layer in range(n_layers):
        pattern = cache[utils.get_act_name("pattern", layer)]  # [batch, head, dest, src]
        # for destination positions in the second half, the "induction" source
        # is (dest - seq_len), i.e. one-past the matching token last time
        diag = pattern.diagonal(offset=-seq_len + 1, dim1=-2, dim2=-1)
        scores[layer] = diag.mean(dim=(0, -1))
    return scores

seq_len = 25
scores = induction_scores(cache, seq_len, model.cfg.n_layers, model.cfg.n_heads)
px.imshow(scores, labels=dict(x="head", y="layer"), title="Induction score by head",
          color_continuous_scale="Blues").show()


In [4]:
# pull out the top few heads and eyeball their attention pattern directly
top = torch.topk(scores.flatten(), 3)
for score, idx in zip(top.values, top.indices):
    layer, head = divmod(idx.item(), model.cfg.n_heads)
    print(f"layer {layer} head {head}: induction score {score:.3f}")


layer 5 head 5: induction score 0.884
layer 7 head 10: induction score 0.849
layer 5 head 1: induction score 0.841


## Visualize the winner

Look at the actual attention pattern of the top-scoring head on the
induction batch — you should see a clear diagonal stripe `seq_len` steps
back from the current position, once you're into the repeated half.


In [5]:
layer, head = divmod(top.indices[0].item(), model.cfg.n_heads)
pattern = cache[utils.get_act_name("pattern", layer)][0, head]
str_tokens = [str(t) for t in tokens[0].cpu().tolist()]
px.imshow(pattern.cpu(), title=f"Layer {layer} Head {head} attention pattern").show()


## Next

`04_logit_lens.ipynb` — instead of looking at *attention patterns*, look at
what each layer's residual stream would predict if the model stopped there.
